In [1]:
import ast
import json
import re
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load the 10k Gurgaon dataset
raw_df = pd.read_csv('gurgaon_10k_appertment_all_rows.csv')

In [3]:
len(raw_df['PropertyName'].unique())

917

In [4]:
raw_df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities,PHOTO_URL,MEDIUM_PHOTO_URL
0,Alpha Corp GurgaonOne 84,4 BHK Flat in Sector 84 Gurgaon,"['Shopping', 'Education', 'Hospital', 'Bank', ...","{'Shopping': '4 Shoppings', 'Education': '2 Ed...",https://www.99acres.com/alpha-corp-gurgaonone-...,"{'4 BHK': {'building_type': 'Apartment', 'area...","['Intercom Facility', 'Lift(s)', 'Water purifi...",https://mediacdn.99acres.com/media1/22409/13/4...,https://mediacdn.99acres.com/media1/22409/13/4...
1,DLF The Ultima,4 BHK Flat in Sector 81 Gurgaon,"[""st. xavier's"", 'apollo miracle']","{""st. xavier's"": '500m', 'apollo miracle': '1km'}",https://www.99acres.com/dlf-the-ultima-sector-...,"{'4 BHK': {'building_type': 'Apartment', 'area...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...",https://mediacdn.99acres.com/media1/22425/6/44...,https://mediacdn.99acres.com/media1/22425/6/44...
2,Emaar MGF The Palm Drive,3 BHK Flat in Sector 66 Gurgaon,"['huda city center', 'DPS School']","{'huda city center': 'Nearby', 'DPS School': '...",https://www.99acres.com/emaar-mgf-the-palm-dri...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Water purifier', 'Security / Fire Alarm', 'P...",https://mediacdn.99acres.com/media1/22386/13/4...,https://mediacdn.99acres.com/media1/22386/13/4...
3,Experion Windchants,3 BHK Flat in Sector 112 Gurgaon,"['Shopping', 'Connectivity', 'Education', 'Hos...","{'Shopping': '1 Shopping', 'Connectivity': '1 ...",https://www.99acres.com/experion-windchants-se...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Power Back-up', 'Feng Shui / Vaastu Complian...",https://mediacdn.99acres.com/media1/22353/14/4...,https://mediacdn.99acres.com/media1/22353/14/4...
4,ATS Triumph,3 BHK Flat in Sector 104 Gurgaon,"['MetroStation', 'Shopping', 'Connectivity', '...","{'MetroStation': '1 Metro Station', 'Shopping'...",https://www.99acres.com/ats-triumph-sector-104...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Power Back-up', 'Intercom Facility', 'Lift(s...",https://mediacdn.99acres.com/media1/22352/4/44...,https://mediacdn.99acres.com/media1/22352/4/44...


In [5]:
raw_df.shape

(10704, 9)

In [6]:
raw_df.isnull().sum()

PropertyName          1569
PropertySubName          0
NearbyLocations         56
LocationAdvantages      56
Link                     0
PriceDetails             0
TopFacilities           82
PHOTO_URL             1840
MEDIUM_PHOTO_URL      1840
dtype: int64

In [7]:
# 1. Drop rows where PropertyName is missing (unnamed independent floors/plots)
raw_df = raw_df.dropna(subset=['PropertyName']).copy()

In [8]:
# 2. Remove monthly rental listings via PriceDetails
def is_rental_listing(price_detail_str):
    try:
        d = ast.literal_eval(str(price_detail_str))
        for bhk, info in d.items():
            p = str(info.get('price-range', '')).strip()
            if not p or 'request' in p.lower():
                return False
            if not re.search(r'\b(cr|crore|l|lac|lakh)\b', p, flags=re.IGNORECASE):
                return True
            if re.search(r'\b(l|lac|lakh)\b', p, flags=re.IGNORECASE) and not re.search(r'\b(cr|crore)\b', p, flags=re.IGNORECASE):
                m = re.search(r'([\d\.]+)', p.replace(',', ''))
                if m and float(m.group(1)) <= 6.5 and bhk not in ['1 BHK', '1 RK', 'Land']:
                    return True
    except Exception:
        pass
    return False

raw_df = raw_df[~raw_df['PriceDetails'].apply(is_rental_listing)].copy()

In [9]:
raw_df['npxid_code'] = raw_df['Link'].str.extract(r'(npxid-[a-zA-Z0-9]+)')

In [10]:
raw_df['npxid_code'].head()

0      npxid-r5096
1      npxid-r7182
3      npxid-r4342
5     npxid-r25860
7    npxid-r368291
Name: npxid_code, dtype: str

In [11]:
NPXID_CANONICAL_NAMES = {
    'npxid-r404070': 'DLF The Arbour',
    'npxid-r400451': 'Elan The Presidential',
    'npxid-r418493': 'Ganga Nandaka',
    'npxid-r436046': 'DLF Privana North',
    'npxid-r411431': 'Emaar Urban Oasis',
    'npxid-r403309': 'Whiteland The Aspen',
    'npxid-r402578': 'SS Cendana Residences',
    'npxid-r405529': 'Navraj The Antalyas',
    'npxid-r405511': 'Oxirich Chintamanis',
    'npxid-r404068': 'M3M Crown',
    'npxid-r405181': 'M3M Golf Hills',
    'npxid-r400415': 'Smartworld One DXP',
    'npxid-r396484': 'Signature Global City 37D Ph 2',
    'npxid-r412628': 'Pareena Hanu Residency',
    'npxid-r413599': 'Ameya Sapphire Residences',
    'npxid-r414174': 'Laburnum Victory Floors',
    'npxid-r12302': 'Kendriya Vihar',
    'npxid-r137273': 'Ganpati Apartment',
    'npxid-r29074': 'CGHS HEWO Apartments',
    'npxid-r270741': 'Signature The Roselia',
    'npxid-r306651': 'Godrej Meridien',
    'npxid-r32692': 'Emaar Emerald Hills',
    'npxid-r330597': 'Signature Global Park',
    'npxid-r343747': 'Suncity Avenue 76',
    'npxid-r27710': 'DLF Wellington Estate',
    'npxid-r136601': 'Sanskriti Apartments',
    'npxid-r26952': 'Ansal Oriental Villas',
}

In [12]:
# Automatically assign the most common clean name for all other npxid project codes
for code, grp in raw_df.dropna(subset=['npxid_code']).groupby('npxid_code'):
    if code not in NPXID_CANONICAL_NAMES:
        NPXID_CANONICAL_NAMES[code] = grp['PropertyName'].mode().iloc[0].strip()

ACRONYMS = ['DLF', 'M3M', 'AIPL', 'BPTP', 'ATS', 'SS', 'JMS', 'ROF', 'CHD', 'GLS', 'MVN', 'AVL', 'NBCC', 'HCBS', 'ILD', 'CGHS', 'RWA', 'AWHO', 'HSIIDC', 'IRWO', 'MGF', 'SBTL', 'MRG']

def resolve_property_name(row):
    code = row['npxid_code']
    if pd.notna(code) and code in NPXID_CANONICAL_NAMES:
        name = NPXID_CANONICAL_NAMES[code]
    else:
        name = str(row['PropertyName']).strip()
        name = re.split(r',\s*|\s{2,}', name)[0].strip()
        name = re.sub(r'\s+Sector\s+\d+[A-Za-z]?\s*(?:Gurgaon|Gurugram)?$', '', name, flags=re.IGNORECASE).strip()

    # Normalize casing if any word starts with a lowercase letter
    if any(w[0].islower() for w in name.split() if w and w[0].isalpha() and w.lower() not in {'at', 'in', 'by', 'of', 'the', 'and', 'on'}):
        name = name.title()
    for acr in ACRONYMS:
        name = re.sub(rf'\b{acr}\b', acr, name, flags=re.IGNORECASE)
    return name

raw_df['PropertyName'] = raw_df.apply(resolve_property_name, axis=1)

In [13]:
JUNK_PROPERTY_PATTERNS = [
    # Placeholders, nulls, generic words, or bare builder/authority names
    r'^(?:nul|null|none|nan|on\s*request|residental|residential|dlf|ansal|ansals\s*api|hsvp.*|housing\s*board.*|huda\s*(?:flats?|floors?)|rwa|sector|inox\s*mall|my\s*home|new\s*era\s*home)$',
    # Generic floor/apartment/plot descriptions without a real society name
    r'^(?:brand\s*new\s*)?(?:super\s*|ultra\s*|optimal\s*)*(?:luxury|luxurious)?\s*(?:independent\s*|private\s*|pvt\.?\s*)?(?:builder\s*|duplex\s*|triplex\s*|new\s*|corner\s*|first\s*|top\s*|palm\s*)*(?:floor|floors|floore|florr|builderfloor|apartment|apartments|flat|flats|house|houes|kothi|plot|plots|residency)\b',
    # Generic directional / facing / feature descriptions
    r'^(?:corner|park\s*facing|east\s*facing|north\s*facing|two\s*side\s*open|top\s*floor|first\s*floor|duplex|triplex|premium\s*construction|24\s*meter|new\s*launch|plot\s*on|independent\s*house|cghs\s*apartments?|residential\s*plots\s*ddjay)\b',
    # Bare locality/sector/block names typed as PropertyName
    r'^(?:sector[\s-]*\d+|sec[\s-]*\d+|dlf\s*(?:city\s*)?(?:builder\s*)?(?:floor|floors|phase)\b|sushant\s*lok\b|south\s*city\b|[a-z]\s*block\b|block\s*[a-z]\b|sohna\s*road|arjun\s*marg|vigyan\s*vihar)',
    # Any name containing generic 'luxury builder floor(s)' / 'private builder floor' / 'builder floore'
    r'\b(?:luxury\s*builder\s*floors?|luxurious\s*builder\s*floors?|ultra\s*luxurious\s*builder\s*floor|luxury\s*dlf\s*city\s*floors?|optimal\s*(?:ultra\s*)?luxury.*floors?|private\s*builder\s*floor|duplex\s*(?:builder\s*)?floors?|landmark\s*builder\s*floors|trehan\s*builder\s*floore|independent\s*flooy|ss\s*group\s*premium\s*luxury\s*floors)\b'
]

def is_junk_property_name(name):
    if pd.isna(name):
        return True
    s = str(name).strip()
    if len(s) < 3:
        return True
    return any(re.search(pat, s, flags=re.IGNORECASE) for pat in JUNK_PROPERTY_PATTERNS)

In [14]:
# Filter out all junk property names
raw_df = raw_df[~raw_df['PropertyName'].apply(is_junk_property_name)].copy()
print(f"Cleaned rows: {len(raw_df)} | Valid Unique Societies: {raw_df['PropertyName'].nunique()}")

Cleaned rows: 8206 | Valid Unique Societies: 635


In [15]:
def safe_parse_list(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    try:
        parsed = ast.literal_eval(str(val))
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    return re.findall(r"'(.*?)'", str(val))

In [16]:
def safe_parse_dict(val):
    if pd.isna(val):
        return {}
    if isinstance(val, dict):
        return val
    try:
        parsed = ast.literal_eval(str(val))
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass
    try:
        return json.loads(str(val).replace("'", '"'))
    except Exception:
        return {}

In [17]:
def merge_top_facilities(series):
    # Keep facilities that appear in at least 20% of the society's listings (or all if <= 3 listings)
    # This prevents a single spam listing with all 34 boxes checked from distorting cosine_sim1
    valid_lists = [safe_parse_list(v) for v in series if safe_parse_list(v)]
    if not valid_lists:
        return "[]"
    counts = Counter(item for sublist in valid_lists for item in sublist)
    threshold = max(1, int(len(valid_lists) * 0.25)) if len(valid_lists) >= 4 else 1
    filtered = [item for item, cnt in counts.most_common() if cnt >= threshold]
    return str(filtered if filtered else list(counts.keys()))

In [18]:
GENERIC_LOC_KEYS = {
    'metrostation', 'shopping', 'connectivity', 'education', 'hospital', 'airport',
    'railwaystation', 'officecomplex', 'hotel', 'amusementpark', 'golfcourse',
    'stadium', 'bank', 'park', 'miscellaneous', 'pharmacy', 'atm', 'restaurant',
    'busstop', 'religiousplace', 'attraction', 'parking', 'library', 'swimmingpool', 'club'
}

def merge_nearby_locations(series):
    near_list, seen = [], set()
    all_items = [item for val in series for item in safe_parse_list(val)]
    for item in all_items:
        if item.lower() not in GENERIC_LOC_KEYS and item not in seen:
            seen.add(item)
            near_list.append(item)
    if not near_list:
        for item in all_items:
            if item not in seen:
                seen.add(item)
                near_list.append(item)
    return str(near_list)

In [19]:
def merge_location_advantages(series):
    merged_loc = {}
    for val in series:
        for k, v in safe_parse_dict(val).items():
            if k not in merged_loc or str(merged_loc[k]).lower() == 'nearby':
                merged_loc[k] = v
    return str(merged_loc)

In [20]:
def extract_areas_from_str(area_str):
    cleaned = str(area_str).replace(',', '').replace('sq.ft.', '').strip()
    return [float(re.search(r'[\d\.]+', p).group(0)) for p in cleaned.split('-') if re.search(r'[\d\.]+', p)]

In [21]:
def extract_prices_in_cr(price_str):
    p_str = str(price_str).strip()
    if not p_str or 'request' in p_str.lower() or not re.search(r'\b(cr|crore|l|lac|lakh)\b', p_str, flags=re.IGNORECASE):
        return []
    is_lac = bool(re.search(r'\b(l|lac|lakh)\b', p_str, flags=re.IGNORECASE)) and not bool(re.search(r'\b(cr|crore)\b', p_str, flags=re.IGNORECASE))
    prices = []
    for part in p_str.split('-'):
        m = re.search(r'[\d\.]+', part.replace(',', ''))
        if m:
            p_val = float(m.group(0))
            if re.search(r'\b(l|lac|lakh)\b', part, flags=re.IGNORECASE) or (is_lac and not re.search(r'\b(cr|crore)\b', part, flags=re.IGNORECASE)):
                p_val /= 100.0
            prices.append(p_val)
    return prices

In [22]:
def format_bhk_summary(pdata):
    areas, prices = pdata['areas'], pdata['prices']
    area_fmt = f"{min(areas):,.0f} - {max(areas):,.0f} sq.ft." if len(areas) > 1 and min(areas) != max(areas) else (f"{areas[0]:,.0f} sq.ft." if areas else "")
    price_fmt = f"₹ {min(prices):.2f} - {max(prices):.2f} Cr" if len(prices) > 1 and min(prices) != max(prices) else (f"₹ {prices[0]:.2f} Cr" if prices else "Price on Request")
    return {
        'building_type': pdata['building_type'],
        'area_type': pdata['area_type'],
        'area': area_fmt,
        'price-range': price_fmt
    }

In [23]:
def merge_price_details(series):
    merged_price = {}
    for val in series:
        for bhk, info in safe_parse_dict(val).items():
            if not isinstance(info, dict):
                continue
            if bhk not in merged_price:
                merged_price[bhk] = {
                    'building_type': info.get('building_type', ''),
                    'area_type': info.get('area_type', ''),
                    'areas': [],
                    'prices': []
                }
            merged_price[bhk]['areas'].extend(extract_areas_from_str(info.get('area', '')))
            merged_price[bhk]['prices'].extend(extract_prices_in_cr(info.get('price-range', '')))
    return str({bhk: format_bhk_summary(pdata) for bhk, pdata in merged_price.items()})

In [24]:
def first_valid(series):
    valid = series.dropna()
    return valid.iloc[0] if not valid.empty else None

In [25]:
df = raw_df.groupby('PropertyName', sort=False, as_index=False).agg({
    'PropertySubName': 'first',
    'NearbyLocations': merge_nearby_locations,
    'LocationAdvantages': merge_location_advantages,
    'Link': 'first',
    'PriceDetails': merge_price_details,
    'TopFacilities': merge_top_facilities,
    'PHOTO_URL': first_valid,
    'MEDIUM_PHOTO_URL': first_valid
})

In [26]:
df.shape

(635, 9)

In [27]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities,PHOTO_URL,MEDIUM_PHOTO_URL
0,Alpha Corp GurgaonOne 84,4 BHK Flat in Sector 84 Gurgaon,"['ARVY HOSPITAL', 'DPS 84', 'dps', 'aarvy']","{'Shopping': '4 Shoppings', 'Education': '2 Ed...",https://www.99acres.com/alpha-corp-gurgaonone-...,"{'4 BHK': {'building_type': 'Apartment', 'area...","['Intercom Facility', 'Lift(s)', 'Maintenance ...",https://mediacdn.99acres.com/media1/22409/13/4...,https://mediacdn.99acres.com/media1/22409/13/4...
1,DLF The Ultima,4 BHK Flat in Sector 81 Gurgaon,"[""st. xavier's"", 'apollo miracle', 'Apollo Mir...","{""st. xavier's"": '500m', 'apollo miracle': '1k...",https://www.99acres.com/dlf-the-ultima-sector-...,"{'4 BHK': {'building_type': 'Apartment', 'area...","['Maintenance Staff', 'Park', 'Intercom Facili...",https://mediacdn.99acres.com/media1/22425/6/44...,https://mediacdn.99acres.com/media1/22425/6/44...
2,Experion Windchants,3 BHK Flat in Sector 112 Gurgaon,"['euro international', 'nora solomon', 'Jai Sa...","{'Shopping': '1 Shopping', 'Connectivity': '1 ...",https://www.99acres.com/experion-windchants-se...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Maintenance Staff', 'Park', 'Feng Shui / Vaa...",https://mediacdn.99acres.com/media1/22353/14/4...,https://mediacdn.99acres.com/media1/22353/14/4...
3,ATS Triumph,3 BHK Flat in Sector 104 Gurgaon,"['DPS SCHOOL', 'MANIPAL HOSPITAL', 'royal oak'...","{'MetroStation': '1 Metro Station', 'Shopping'...",https://www.99acres.com/ats-triumph-sector-104...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Intercom Facility', 'Lift(s)', 'Maintenance ...",https://mediacdn.99acres.com/media1/22352/4/44...,https://mediacdn.99acres.com/media1/22352/4/44...
4,M3M Skywalk,2 BHK Flat in Sector 74 Gurgaon,"['DPS', 'M3M', 'huda city center', 'suraj scho...","{'MetroStation': '1 Metro Station', 'Shopping'...",https://www.99acres.com/m3m-skywalk-sector-74-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Intercom Facility', 'Lift(s)', 'Park', 'Swim...",https://mediacdn.99acres.com/media1/22348/8/44...,https://mediacdn.99acres.com/media1/22348/8/44...


In [28]:
print("NearbyLocations:", df.iloc[1].NearbyLocations)

NearbyLocations: ["st. xavier's", 'apollo miracle', 'Apollo Miracle Hospital', 'Vatika Town Square 2', 'DPS international school', 'Rockland hospital', 'St Xavier', 'arvy Hospital', 'Saphire 83', 'SS OMNIA', 'ST. XAVIERS SCHOOL', 'MIRACLES MEDICLINIC', 'St. xavier school DPS', 'DPS,RPS,MAITRIKIRAN,ST.XAVIOUR', 'APOLLO,AARVY,ROCKLAND', 'APOLLO,ROCKLAND,AARVY', 'ARVY HOSPITAL', 'SAPPHIRE 83', 'ST. XAVIERS', 'St Xaviors School', 'Sideshwar', 'Appolo Miracle', 'MIRACLES APOLLO HOSPITAL', 'TOWN SQURE 2', 'SIDHESHWAR SCHOOL', 'MEDANTA HOSPITAL', 'SAME COMPAUND', 'DPS', 'Huda Metro Station', 'Miracles Apollo Cradle', 'huda city center', 'Delhi Public School']


In [29]:
print("LocationAdvantages:", df.iloc[1].LocationAdvantages)

LocationAdvantages: {"st. xavier's": '500m', 'apollo miracle': '1km', 'MetroStation': '1 Metro Station', 'Shopping': '1 Shopping', 'Connectivity': '2 Connectivities', 'Education': '2 Educations', 'Hospital': '1 Hospital', 'Airport': '1 Airport', 'RailwayStation': '1 Railway Station', 'Apollo Miracle Hospital': '500m', 'Vatika Town Square 2': '500m', 'DPS international school': 'Nearby', 'Rockland hospital': 'Nearby', 'St Xavier': '1km', 'arvy Hospital': '1km', 'Saphire 83': '1km', 'SS OMNIA': '500m', 'ST. XAVIERS SCHOOL': '250m', 'MIRACLES MEDICLINIC': 'Nearby', 'St. xavier school DPS': 'Nearby', 'DPS,RPS,MAITRIKIRAN,ST.XAVIOUR': 'Nearby', 'APOLLO,AARVY,ROCKLAND': 'Nearby', 'APOLLO,ROCKLAND,AARVY': 'Nearby', 'ARVY HOSPITAL': '1km', 'SAPPHIRE 83': '1km', 'ST. XAVIERS': 'Nearby', 'St Xaviors School': '500m', 'Sideshwar': '250m', 'Appolo Miracle': '250m', 'MIRACLES APOLLO HOSPITAL': '250m', 'TOWN SQURE 2': '500m', 'SIDHESHWAR SCHOOL': 'Nearby', 'MEDANTA HOSPITAL': '3km', 'SAME COMPAUND': 

In [30]:
print("PriceDetails:", df.iloc[1].PriceDetails)

PriceDetails: {'4 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '2,705 - 3,035 sq.ft.', 'price-range': '₹ 3.45 - 4.25 Cr'}, '3 BHK': {'building_type': 'Apartment', 'area_type': 'Super Built-up Area', 'area': '2,032 - 2,132 sq.ft.', 'price-range': '₹ 2.20 - 2.75 Cr'}}


In [31]:
print("TopFacilities:", df.iloc[1].TopFacilities)

TopFacilities: ['Maintenance Staff', 'Park', 'Intercom Facility', 'Lift(s)', 'Swimming Pool', 'Security Personnel', 'Internet/wi-fi connectivity', 'Shopping Centre', 'Fitness Centre / GYM', 'Rain Water Harvesting', 'Club house / Community Center', 'Wheelchair Friendly', 'Pet Friendly', 'Natural Light', 'Visitor Parking', 'Power Back-up', 'Separate entry for servant room', 'In a Gated Society', 'No open drainage around', 'Security / Fire Alarm', 'Piped-gas', 'Spacious Interiors', 'Water Storage', 'Water softening plant', 'Airy Rooms', 'Waste Disposal', 'Feng Shui / Vaastu Compliant', 'High Ceiling Height', 'Centrally Air Conditioned', 'Water purifier', 'False Ceiling Lighting', 'Private Garden / Terrace', 'Low Density Society']


In [32]:
def extract_list(s):
    if isinstance(s, list):
        return s
    try:
        parsed = ast.literal_eval(str(s))
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed]
    except Exception:
        pass
    return re.findall(r"'(.*?)'", str(s))

df['TopFacilities'] = df['TopFacilities'].apply(extract_list)
df['FacilitiesStr'] = df['TopFacilities'].apply(' '.join)
df['FacilitiesStr'][0]

'Intercom Facility Lift(s) Maintenance Staff Swimming Pool Security Personnel Shopping Centre Fitness Centre / GYM Club house / Community Center No open drainage around Pet Friendly Internet/wi-fi connectivity Waste Disposal Water purifier Park Rain Water Harvesting Water softening plant Separate entry for servant room Natural Light Visitor Parking Wheelchair Friendly False Ceiling Lighting Spacious Interiors Water Storage Power Back-up High Ceiling Height Centrally Air Conditioned Private Garden / Terrace Near Bank Airy Rooms Security / Fire Alarm Feng Shui / Vaastu Compliant Piped-gas Low Density Society'

In [33]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(df['FacilitiesStr'])
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim1.shape

(635, 635)

In [34]:
def recommend_properties(property_name, cosine_sim=cosine_sim1):
    matches = df.index[df['PropertyName'].str.lower() == property_name.lower()].tolist()
    if not matches:
        matches = df.index[df['PropertyName'].str.contains(property_name, case=False, na=False)].tolist()
    if not matches:
        raise ValueError(f"Property '{property_name}' not found.")
    idx = matches[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]

    property_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]

    recommendations_df = pd.DataFrame({
        'PropertyName': df['PropertyName'].iloc[property_indices].values,
        'SimilarityScore': scores
    })
    return recommendations_df

recommend_properties("DLF The Arbour")

,PropertyName,SimilarityScore
0,DLF Royalton Tower,0.530518
1,Pivotal Riddhi Siddhi,0.525935
2,Wembley Premium Tower,0.515324
3,M3M Corner Walk,0.500077
4,Laburnum Apartment,0.500077


## PriceDetails

In [35]:
def parse_detail_dict(detail_str):
    """Safely parse PriceDetails string into a Python dictionary."""
    try:
        details = ast.literal_eval(str(detail_str))
        if isinstance(details, dict):
            return details
        return json.loads(str(detail_str).replace("'", '"'))
    except Exception:
        return {}

In [36]:
def parse_area_range(area_str):
    """Extract (area_low, area_high) in sq.ft. from an area string."""
    if not area_str:
        return None, None

    clean_area = str(area_str).replace(',', '').replace('sq.ft.', '').replace('sq.ft', '')
    nums = [
        float(m.group(0))
        for part in clean_area.split('-')
        if (m := re.search(r'[\d\.]+', part))
    ]

    if len(nums) == 1:
        return nums[0], nums[0]
    elif len(nums) >= 2:
        return min(nums), max(nums)
    return None, None

In [37]:
def parse_price_range(price_str):
    """Extract (price_low, price_high) in Crores (Cr) from a price string."""
    if not price_str or 'request' in str(price_str).lower():
        return None, None

    price_str = str(price_str)
    overall_is_lac = (
        bool(re.search(r'\b(l|lac|lakh)\b', price_str, re.IGNORECASE))
        and not bool(re.search(r'\b(cr|crore)\b', price_str, re.IGNORECASE))
    )

    nums = []
    for part in price_str.split('-'):
        m = re.search(r'[\d\.]+', part.replace(',', ''))
        if m:
            val = float(m.group(0))
            is_part_lac = re.search(r'\b(l|lac|lakh)\b', part, re.IGNORECASE)
            is_part_cr = re.search(r'\b(cr|crore)\b', part, re.IGNORECASE)
            if is_part_lac or (overall_is_lac and not is_part_cr):
                val /= 100.0
            nums.append(val)

    if len(nums) == 1:
        return nums[0], nums[0]
    elif len(nums) >= 2:
        return min(nums), max(nums)
    return None, None

In [38]:
def refined_parse_modified_v2(detail_str):
    """Extract building_type, area range, and price range for all BHK configurations."""
    details = parse_detail_dict(detail_str)
    extracted = {}

    for bhk, detail in details.items():
        if not isinstance(detail, dict):
            continue
        extracted[f'building type_{bhk}'] = detail.get('building_type')
        extracted[f'area low {bhk}'], extracted[f'area high {bhk}'] = parse_area_range(detail.get('area', ''))
        extracted[f'price low {bhk}'], extracted[f'price high {bhk}'] = parse_price_range(detail.get('price-range', ''))

    return extracted

In [39]:
def build_price_features_df(df):
    """Build the 40-column price and area feature DataFrame indexed by PropertyName."""
    configs = ['1 BHK', '2 BHK', '3 BHK', '4 BHK', '5 BHK', '6 BHK', '1 RK', 'Land']
    data_refined = []

    for _, row in df.iterrows():
        features = refined_parse_modified_v2(row['PriceDetails'])
        new_row = {'PropertyName': row['PropertyName']}
        for config in configs:
            new_row[f'building type_{config}'] = features.get(f'building type_{config}')
            new_row[f'area low {config}'] = features.get(f'area low {config}')
            new_row[f'area high {config}'] = features.get(f'area high {config}')
            new_row[f'price low {config}'] = features.get(f'price low {config}')
            new_row[f'price high {config}'] = features.get(f'price high {config}')
        data_refined.append(new_row)

    df_out = pd.DataFrame(data_refined).set_index('PropertyName')
    df_out['building type_Land'] = df_out['building type_Land'].replace({'': 'Land'})
    return df_out

In [40]:
df_final_refined_v2 = build_price_features_df(df)
df_final_refined_v2.head()

,building type_1 BHK,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,building type_2 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,...,building type_1 RK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,building type_Land,area low Land,area high Land,price low Land,price high Land
PropertyName,,,,,,,,,,,,,,,,,,,,,
Alpha Corp GurgaonOne 84,NaN,NaN,NaN,NaN,NaN,Apartment,1270.0,1534.0,0.85,1.25,...,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN
DLF The Ultima,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN
Experion Windchants,NaN,NaN,NaN,NaN,NaN,Apartment,2441.0,2441.0,2.89,3.00,...,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN
ATS Triumph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN
M3M Skywalk,NaN,NaN,NaN,NaN,NaN,Apartment,1400.0,1426.0,1.45,1.82,...,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN


In [41]:
categorical_columns = df_final_refined_v2.select_dtypes(include=['object']).columns.tolist()
ohe_df = pd.get_dummies(df_final_refined_v2, columns=categorical_columns, drop_first=True)
ohe_df.fillna(0, inplace=True)

C:\Users\prama\AppData\Local\Temp\ipykernel_9416\1642634783.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df_final_refined_v2.select_dtypes(include=['object']).columns.tolist()


,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,...,building type_2 BHK_Independent Floor,building type_2 BHK_Villa,building type_3 BHK_Independent Floor,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Independent Floor,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,
Alpha Corp GurgaonOne 84,0.0,0.0,0.0,0.0,1270.0,1534.0,0.85,1.25,1923.0,2244.0,...,False,False,False,False,False,False,False,False,False,False
DLF The Ultima,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,2032.0,2132.0,...,False,False,False,False,False,False,False,False,False,False
Experion Windchants,0.0,0.0,0.0,0.0,2441.0,2441.0,2.89,3.00,2802.0,3763.0,...,False,False,False,False,False,False,False,True,False,False
ATS Triumph,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,2290.0,2290.0,...,False,False,False,False,False,False,False,False,False,False
M3M Skywalk,0.0,0.0,0.0,0.0,1400.0,1426.0,1.45,1.82,2000.0,2074.0,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Spaze Kalistaa,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,3200.0,3200.0,...,False,False,False,False,False,False,False,False,False,False
Mayfield Palm Floors,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,...,False,False,False,False,True,False,False,False,False,False
Narkanda Apartment,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,1800.0,1800.0,...,False,False,False,False,False,False,False,False,False,False


In [42]:
ohe_df

,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,...,building type_2 BHK_Independent Floor,building type_2 BHK_Villa,building type_3 BHK_Independent Floor,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Independent Floor,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,
Alpha Corp GurgaonOne 84,0.0,0.0,0.0,0.0,1270.0,1534.0,0.85,1.25,1923.0,2244.0,...,False,False,False,False,False,False,False,False,False,False
DLF The Ultima,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,2032.0,2132.0,...,False,False,False,False,False,False,False,False,False,False
Experion Windchants,0.0,0.0,0.0,0.0,2441.0,2441.0,2.89,3.00,2802.0,3763.0,...,False,False,False,False,False,False,False,True,False,False
ATS Triumph,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,2290.0,2290.0,...,False,False,False,False,False,False,False,False,False,False
M3M Skywalk,0.0,0.0,0.0,0.0,1400.0,1426.0,1.45,1.82,2000.0,2074.0,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Spaze Kalistaa,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,3200.0,3200.0,...,False,False,False,False,False,False,False,False,False,False
Mayfield Palm Floors,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,...,False,False,False,False,True,False,False,False,False,False
Narkanda Apartment,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,1800.0,1800.0,...,False,False,False,False,False,False,False,False,False,False


In [43]:
scaler = StandardScaler()
ohe_df_normalized = pd.DataFrame(
    scaler.fit_transform(ohe_df),
    columns=ohe_df.columns,
    index=ohe_df.index
)

cosine_sim2 = cosine_similarity(ohe_df_normalized)
cosine_sim2.shape

(635, 635)

In [44]:
def recommend_properties_with_scores(property_name, top_n=10):
    if property_name not in ohe_df_normalized.index:
        matches = [idx for idx in ohe_df_normalized.index if property_name.lower() in idx.lower()]
        if not matches:
            raise ValueError(f"Property '{property_name}' not found.")
        property_name = matches[0]

    sim_scores = list(enumerate(cosine_sim2[ohe_df_normalized.index.get_loc(property_name)]))
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    top_properties = ohe_df_normalized.index[top_indices].tolist()

    return pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })

recommend_properties_with_scores('M3M Golf Hills')

,PropertyName,SimilarityScore
0,Unitech Fresco,0.995955
1,Parsvnath Green Ville,0.988445
2,Capital Residences 360,0.980962
3,M3M Flora,0.977220
4,Unitech The Residences,0.976212
5,Ansal Sushant Apartments,0.974853
6,M3M Antalya Hills,0.967349
7,DLF Princeton Estate,0.963275
8,M3M Escala,0.961377
9,M3M Sierra 68,0.959274


In [45]:
GENERIC_CATEGORY_KEYS = {
    'metrostation', 'shopping', 'connectivity', 'education', 'hospital', 'airport',
    'railwaystation', 'officecomplex', 'hotel', 'amusementpark', 'golfcourse',
    'stadium', 'bank', 'park', 'miscellaneous', 'pharmacy', 'atm', 'restaurant',
    'busstop', 'religiousplace', 'attraction', 'parking', 'library', 'swimmingpool',
    'club', 'mtro', 'schl'
}

In [46]:
CANONICAL_LOCATION_PATTERNS = [
    # Airports
    (r'\b(indira gandhi|igi\b|igia\b|i\.g\.i|delhi airport|airport t3|terminal 3)', 'Indira Gandhi International Airport'),
    # Highways & Expressways
    (r'\b(nh\s*[-:]?\s*8|nh\s*[-:]?\s*48|national highway\s*8|national highway\s*48|delhi jaipur expressway|jaipur highway)', 'National Highway 8'),
    (r'\b(dwarka exp|dwarka express|northern peripheral)', 'Dwarka Expressway'),
    (r'\b(golf course ext|golf course extension)', 'Golf Course Extension Road'),
    (r'\b(golf course road|golf course rd)', 'Golf Course Road'),
    (r'\b(sohna road|sohna rd|badshahpur sohna)', 'Sohna Road'),
    (r'\b(mg road|mehrauli[\s-]*gurgaon|m\.g\.\s*road)', 'Mehrauli-Gurgaon Road'),
    (r'\b(southern peripheral|spr\b|spr road)', 'Southern Peripheral Road'),
    (r'\b(kmp\b|kundli manesar)', 'KMP Expressway'),
    (r'\b(pataudi road|pataudi rd)', 'Pataudi Road'),
    (r'\b(bajghera road|bajghera rd)', 'Bajghera Road'),
    # Metro & Railway Stations
    (r'\b(huda city|hudda city|huda metro|hudda metro|millennium city centre|\bhuda\b|\bhudda\b)', 'HUDA City Centre Metro Station'),
    (r'\b(dwarka sec.*21|dwarka metro)', 'Dwarka Sector 21 Metro Station'),
    (r'\b(sector 55[\s/-]*56|sector 56.*metro|rapid metro.*56|\bsector 56\b)', 'Rapid Metro Station Sector 56'),
    (r'\b(sector 53[\s/-]*54|sector 54.*metro|\bsector 54\b)', 'Sector 53-54 Rapid Metro Station'),
    (r'\b(rapid metro|\brapid\b)', 'Rapid Metro Gurgaon'),
    (r'\b(iffco chowk)', 'IFFCO Chowk Metro Station'),
    (r'\b(mg road metro)', 'MG Road Metro Station'),
    (r'\b(sikanderpur|sikandarpur)', 'Sikanderpur Metro Station'),
    (r'\b(palam vihar halt)', 'Palam Vihar Halt Railway Station'),
    (r'\b(gurgaon railway|gurugram railway|railway station gurgaon)', 'Gurgaon Railway Station'),
    (r'\b(basai dhankot)', 'Basai Dhankot Railway Station'),
    # Hospitals
    (r'\b(medanta|the medicity)', 'Medanta - The Medicity'),
    (r'\b(artemis)', 'Artemis Hospital'),
    (r'\b(fortis)', 'Fortis Memorial Research Institute'),
    (r'\b(paras hospital|\bparas\b(?!.*trinity))', 'Paras Hospital'),
    (r'\b(max hospital)', 'Max Hospital'),
    (r'\b(park hospital)', 'Park Hospital'),
    (r'\b(manipal hospital|columbia asia|\bcollumbia\b)', 'Manipal Hospital'),
    (r'\b(aarvy hospital|aarvy healthcare)', 'Aarvy Hospital'),
    (r'\b(ck birla)', 'CK Birla Hospital'),
    (r'\b(narayana superspeciality|narayana hospital)', 'Narayana Superspeciality Hospital'),
    (r'\b(w pratiksha|pratiksha hospital)', 'W Pratiksha Hospital'),
    (r'\b(signature hospital)', 'Signature Hospital'),
    (r'\b(apollo miracle|apollo hospital|apollo cradle)', 'Apollo Hospital'),
    (r'\b(nora solomon|nora medical)', 'Nora Solomon Medicenter'),
    (r'\b(vibrant hospital)', 'Vibrant Hospital'),
    (r'\b(rions hospital)', 'Rions Hospital'),
    # Schools & Universities
    (r'\b(dpsg|dps palam)', 'DPSG Palam Vihar'),
    (r'\b(dps international)', 'DPS International School'),
    (r'\b(dps\b|delhi public school)', 'Delhi Public School'),
    (r'\b(gd goenka university)', 'GD Goenka University'),
    (r'\b(gd goenka|g\.d\.\s*goenka)', 'GD Goenka Public School'),
    (r'\b(heritage xperiential|heritage school)', 'Heritage Xperiential Learning School'),
    (r'\b(shri ram school|tsrs)', 'The Shri Ram School'),
    (r'\b(st\.?\s*xavier|st\.?\s*xavious)', "St. Xavier's High School"),
    (r'\b(scottish high)', 'Scottish High International School'),
    (r'\b(suncity school)', 'Suncity School'),
    (r'\b(lotus valley)', 'Lotus Valley International School'),
    (r'\b(amity international|amity school)', 'Amity International School'),
    (r'\b(amity university)', 'Amity University'),
    (r'\b(northcap university|itm university)', 'The NorthCap University'),
    (r'\b(kr mangalam|k\.r\.\s*mangalam)', 'KR Mangalam University'),
    (r'\b(rps\b|rps school|rps international)', 'RPS International School'),
    (r'\b(euro international|euro interntional)', 'Euro International School'),
    (r'\b(mount carmel|mount carmal)', 'Mount Carmel School'),
    (r'\b(pathways)', 'Pathways World School'),
    (r'\b(yaduvanshi)', 'Yaduvanshi Shiksha Niketan'),
    (r'\b(shikshiyan)', 'The Shikshiyan School'),
    (r'\b(pranavananda)', 'Pranavananda International School'),
    # Malls, Hubs & Recreation
    (r'\b(fun n food)', 'Fun N Food WaterPark'),
    (r'\b(ambience mall)', 'Ambience Mall'),
    (r'\b(mgf metropolitan)', 'MGF Metropolitan Mall'),
    (r'\b(sahara mall)', 'Sahara Mall'),
    (r'\b(ardee mall)', 'Ardee Mall'),
    (r'\b(sapphire 83|saphire 83)', 'Sapphire 83 Mall'),
    (r'\b(paras trinity)', 'Paras Trinity Mall Sector 63'),
    (r'\b(pacific d21)', 'Pacific D21 Mall'),
    (r'\b(conscient one)', 'Conscient One Mall'),
    (r'\b(cyber hub|cyber city|dlf cyber)', 'DLF CyberHub'),
    (r'\b(tau devilal|tau devi lal)', 'Tau DeviLal Sports Complex'),
    (r'\b(teri golf)', 'TERI Golf Course'),
    (r'\b(hamoni golf)', 'Hamoni Golf Camp'),
    (r'\b(imt manesar)', 'IMT Manesar'),
    (r'\b(huda market)', 'HUDA Market')
]

In [47]:
def clean_location_name(loc_name):
    if not loc_name or not isinstance(loc_name, str):
        return None
    cleaned = re.sub(r'\s+', ' ', loc_name).strip()
    lower = cleaned.lower()
    if re.sub(r'[^a-z]', '', lower) in GENERIC_CATEGORY_KEYS:
        return None
    lower_no_dist = re.sub(r'\b\d+(?:\.\d+)?\s*(?:km|m|meter|meters)\b', '', lower).strip()
    for pattern, canonical in CANONICAL_LOCATION_PATTERNS:
        if re.search(pattern, lower_no_dist):
            return canonical
    standardized = re.sub(r'\bRd\.?\b', 'Road', cleaned, flags=re.IGNORECASE)
    standardized = re.sub(r'\bSec\.?\b', 'Sector', standardized, flags=re.IGNORECASE)
    standardized = re.sub(r'\bExpy\.?\b', 'Expressway', standardized, flags=re.IGNORECASE)
    standardized = re.sub(r'\bInt\.?\b', 'International', standardized, flags=re.IGNORECASE)
    return standardized.title()

In [48]:
def distance_to_meters(distance_str, key_str=''):
    if distance_str is None or pd.isna(distance_str):
        return None
    s = str(distance_str).strip()
    # Skip category counts like '4 Shoppings', '2 Educations'
    if re.match(r'^\d+\s+(?!km|kilometer|m\b|meter|min|hr)[a-zA-Z]+', s, flags=re.IGNORECASE):
        return None
    # Match compact or spaced units: '500m', '1km', '1.5km', '800 Meter', '2.5 KM'
    m = re.search(r'([\d\.]+)\s*(km|kilometer|kilometre|m\b|meter|metre)', s, flags=re.IGNORECASE)
    if m:
        val = float(m.group(1))
        return val * 1000.0 if m.group(2).lower().startswith('k') else val
    if s.lower() == 'nearby':
        m_key = re.search(r'([\d\.]+)\s*(km|m\b|meter)', str(key_str), flags=re.IGNORECASE)
        if m_key:
            val = float(m_key.group(1))
            return val * 1000.0 if m_key.group(2).lower().startswith('k') else val
        return 500.0
    return None

In [49]:
location_matrix = {}
for index, row in df.iterrows():
    distances = {}
    for location, distance in ast.literal_eval(row['LocationAdvantages']).items():
        dist_m = distance_to_meters(distance, key_str=location)
        if dist_m is not None:
            canon_loc = clean_location_name(location)
            if canon_loc:
                if canon_loc not in distances or dist_m < distances[canon_loc]:
                    distances[canon_loc] = dist_m
    location_matrix[index] = distances

location_df = pd.DataFrame.from_dict(location_matrix, orient='index')
location_df = location_df.reindex(df.index)
location_df.index = df.PropertyName

In [50]:
# Collapse any remaining duplicate column names by minimum distance
location_df = location_df.T.groupby(level=0).min().T
location_df.fillna(54000, inplace=True)
location_df.head()

,"10 Minis Away S""T Xaviour High",100M,15 Shops Inside Society,2-3 School,2-3 School Near Bye,2Km,3-4 School,3-4 School Near Bye,32Nd Avenue,37C,...,Yadhuvanshi,"Yadhuvanshi ,Parvananda",Yadhuvanshi And Parvanada,Yadhuvanshi And Parvananda,Yadhuvanshi School,Yadhuvansi School,Yaduvanshi Shiksha Niketan,Yadvanshi Public School,Yauvanshi,Yudvanshi School
PropertyName,,,,,,,,,,,,,,,,,,,,,
Alpha Corp GurgaonOne 84,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
DLF The Ultima,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
Experion Windchants,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
ATS Triumph,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0
M3M Skywalk,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0


In [51]:
scaler = StandardScaler()
location_df_normalized = pd.DataFrame(
    scaler.fit_transform(location_df),
    columns=location_df.columns,
    index=location_df.index
)

cosine_sim3 = cosine_similarity(location_df_normalized)
cosine_sim3.shape

(635, 635)

In [64]:
def recommend_properties_with_scores(property_name, top_n=10):
    cosine_sim_matrix = 30 * cosine_sim1 + 20 * cosine_sim2 + 8 * cosine_sim3

    if property_name not in location_df_normalized.index:
        matches = [idx for idx in location_df_normalized.index if property_name.lower() in idx.lower()]
        if not matches:
            raise ValueError(f"Property '{property_name}' not found.")
        property_name = matches[0]

    sim_scores = list(enumerate(cosine_sim_matrix[location_df_normalized.index.get_loc(property_name)]))
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    top_properties = location_df_normalized.index[top_indices].tolist()

    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    return recommendations_df

recommend_properties_with_scores('Vatika Gurgaon 21')

,PropertyName,SimilarityScore
0,Vatika City Homes,31.561546
1,Shree Vardhman Flora,31.537111
2,Ashiana Mulberry,30.730326
3,Silverglades The Melia,30.582867
4,Raheja Vedaanta,30.105409
5,AIPL Zen Residences,29.761756
6,Spaze Privy AT4,29.654990
7,Corona Gracieux,29.617604
8,Umang Monsoon Breeze,29.589010
9,Pareena Mi Casa,29.569320


In [63]:
location_df[location_df['Sanklap'] < 22000]

,"10 Minis Away S""T Xaviour High",100M,15 Shops Inside Society,2-3 School,2-3 School Near Bye,2Km,3-4 School,3-4 School Near Bye,32Nd Avenue,37C,...,Yadhuvanshi,"Yadhuvanshi ,Parvananda",Yadhuvanshi And Parvanada,Yadhuvanshi And Parvananda,Yadhuvanshi School,Yadhuvansi School,Yaduvanshi Shiksha Niketan,Yadvanshi Public School,Yauvanshi,Yudvanshi School
PropertyName,,,,,,,,,,,,,,,,,,,,,
Vatika Gurgaon 21,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,...,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0,54000.0


In [54]:
location_df.columns.to_list()

['10 Minis Away S"T Xaviour High',
 '100M',
 '15 Shops Inside Society',
 '2-3 School',
 '2-3 School Near Bye',
 '2Km',
 '3-4 School',
 '3-4 School Near Bye',
 '32Nd Avenue',
 '37C',
 '3Roads The Reach',
 '42 43',
 '4Km',
 '53-54 Metro',
 '54 55',
 '55 56',
 '55-56',
 '55-56 Metro',
 '55-56 Repid Metro',
 '56 Market',
 '56 Metero Station',
 '56 Metr0 Station',
 '56 Metro',
 '56 Metro Sation',
 '56Metro Sation',
 '65Th Avenue',
 'Aaarvy',
 'Aarave Hospitl Sector 90',
 'Aartimish',
 'Aarvi ,Rockland',
 'Aarvy',
 'Aarvy ,Silver Strik',
 'Aarvy And Silver',
 'Aarvy Health Care',
 'Aarvy Hoapital',
 'Aarvy Hosoital',
 'Aarvy Hospital',
 'Aarvy Multi Specialty',
 'Aarvy School',
 'Aarvy, Apollo ,Rockland',
 'Aarvy,Apollo,Rockland',
 'Aarvy,Rockland,Silver Streak',
 'Aary Hospital',
 'Aavry Hospital',
 'Adani',
 'Adarsh Senior Secondary School',
 'Adjacent Suncity Inter',
 'Aimety',
 'Aipl Joy Street',
 'Airia Mall',
 'Airia Mall And Worldmark Mall',
 'Alchemist',
 'Alchemist Hospital',
 'All 

In [55]:
print(df[df['PropertyName'] == 'Tata Gurgaon Gateway']['Link'])

360    https://www.99acres.com/tata-gurgaon-gateway-s...
Name: Link, dtype: str


In [56]:
df.drop(columns= ['MEDIUM_PHOTO_URL'], inplace= True)

C:\Users\prama\AppData\Local\Temp\ipykernel_9416\2741413742.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns= ['MEDIUM_PHOTO_URL'], inplace= True)


In [57]:
df.shape

(635, 9)

In [58]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities,PHOTO_URL,FacilitiesStr
0,Alpha Corp GurgaonOne 84,4 BHK Flat in Sector 84 Gurgaon,"['ARVY HOSPITAL', 'DPS 84', 'dps', 'aarvy']","{'Shopping': '4 Shoppings', 'Education': '2 Ed...",https://www.99acres.com/alpha-corp-gurgaonone-...,"{'4 BHK': {'building_type': 'Apartment', 'area...","[Intercom Facility, Lift(s), Maintenance Staff...",https://mediacdn.99acres.com/media1/22409/13/4...,Intercom Facility Lift(s) Maintenance Staff Sw...
1,DLF The Ultima,4 BHK Flat in Sector 81 Gurgaon,"[""st. xavier's"", 'apollo miracle', 'Apollo Mir...","{""st. xavier's"": '500m', 'apollo miracle': '1k...",https://www.99acres.com/dlf-the-ultima-sector-...,"{'4 BHK': {'building_type': 'Apartment', 'area...","[Maintenance Staff, Park, Intercom Facility, L...",https://mediacdn.99acres.com/media1/22425/6/44...,Maintenance Staff Park Intercom Facility Lift(...
2,Experion Windchants,3 BHK Flat in Sector 112 Gurgaon,"['euro international', 'nora solomon', 'Jai Sa...","{'Shopping': '1 Shopping', 'Connectivity': '1 ...",https://www.99acres.com/experion-windchants-se...,"{'3 BHK': {'building_type': 'Apartment', 'area...","[Maintenance Staff, Park, Feng Shui / Vaastu C...",https://mediacdn.99acres.com/media1/22353/14/4...,Maintenance Staff Park Feng Shui / Vaastu Comp...
3,ATS Triumph,3 BHK Flat in Sector 104 Gurgaon,"['DPS SCHOOL', 'MANIPAL HOSPITAL', 'royal oak'...","{'MetroStation': '1 Metro Station', 'Shopping'...",https://www.99acres.com/ats-triumph-sector-104...,"{'3 BHK': {'building_type': 'Apartment', 'area...","[Intercom Facility, Lift(s), Maintenance Staff...",https://mediacdn.99acres.com/media1/22352/4/44...,Intercom Facility Lift(s) Maintenance Staff Sw...
4,M3M Skywalk,2 BHK Flat in Sector 74 Gurgaon,"['DPS', 'M3M', 'huda city center', 'suraj scho...","{'MetroStation': '1 Metro Station', 'Shopping'...",https://www.99acres.com/m3m-skywalk-sector-74-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","[Intercom Facility, Lift(s), Park, Swimming Po...",https://mediacdn.99acres.com/media1/22348/8/44...,Intercom Facility Lift(s) Park Swimming Pool P...


In [59]:
df1 = pd.read_csv('photo_link.csv')

In [60]:
df1.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities,PHOTO_URL,FacilitiesStr
0,Alpha Corp GurgaonOne 84,4 BHK Flat in Sector 84 Gurgaon,"['ARVY HOSPITAL', 'DPS 84', 'dps', 'aarvy']","{'Shopping': '4 Shoppings', 'Education': '2 Ed...",https://www.99acres.com/alpha-corp-gurgaonone-...,"{'4 BHK': {'building_type': 'Apartment', 'area...","['Intercom Facility', 'Lift(s)', 'Maintenance ...",https://images.unsplash.com/photo-1545324418-c...,Intercom Facility Lift(s) Maintenance Staff Sw...
1,DLF The Ultima,4 BHK Flat in Sector 81 Gurgaon,"[""st. xavier's"", 'apollo miracle', 'Apollo Mir...","{""st. xavier's"": '500m', 'apollo miracle': '1k...",https://www.99acres.com/dlf-the-ultima-sector-...,"{'4 BHK': {'building_type': 'Apartment', 'area...","['Maintenance Staff', 'Park', 'Intercom Facili...",https://images.unsplash.com/photo-151526348799...,Maintenance Staff Park Intercom Facility Lift(...
2,Experion Windchants,3 BHK Flat in Sector 112 Gurgaon,"['euro international', 'nora solomon', 'Jai Sa...","{'Shopping': '1 Shopping', 'Connectivity': '1 ...",https://www.99acres.com/experion-windchants-se...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Maintenance Staff', 'Park', 'Feng Shui / Vaa...",https://images.unsplash.com/photo-152270832359...,Maintenance Staff Park Feng Shui / Vaastu Comp...
3,ATS Triumph,3 BHK Flat in Sector 104 Gurgaon,"['DPS SCHOOL', 'MANIPAL HOSPITAL', 'royal oak'...","{'MetroStation': '1 Metro Station', 'Shopping'...",https://www.99acres.com/ats-triumph-sector-104...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Intercom Facility', 'Lift(s)', 'Maintenance ...",https://images.unsplash.com/photo-148415421896...,Intercom Facility Lift(s) Maintenance Staff Sw...
4,M3M Skywalk,2 BHK Flat in Sector 74 Gurgaon,"['DPS', 'M3M', 'huda city center', 'suraj scho...","{'MetroStation': '1 Metro Station', 'Shopping'...",https://www.99acres.com/m3m-skywalk-sector-74-...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Intercom Facility', 'Lift(s)', 'Park', 'Swim...",https://images.unsplash.com/photo-1560185008-b...,Intercom Facility Lift(s) Park Swimming Pool P...


In [61]:
len(df1['PHOTO_URL'].unique())

635

In [62]:
# import joblib
# joblib.dump(df[['PropertyName', 'Link']], 'see_property_link.pkl')
# joblib.dump(df1[['PropertyName', 'PHOTO_URL']], 'link_of_property_photo.pkl')
# joblib.dump(location_df, 'locality_df.pkl')
# joblib.dump(cosine_sim1, 'cosine_sim1.pkl')
# joblib.dump(cosine_sim2, 'cosine_sim2.pkl')
# joblib.dump(cosine_sim3, 'cosine_sim3.pkl')